# Camera Pose Tester

Move the DOFBot to joint angles (degrees), watch the camera live, and save the poses that give good views of:
- **dealer** cards, **mycards**, **mychips**, **bet** area

Run the cells top to bottom. Set the sliders (click a number to type it exactly) and hit **Move**. Saved poses go to `poses.json` next to this notebook; snapshots go to `snaps/`.

In [ ]:
import json, os, threading, time
import cv2
import ipywidgets as widgets
from IPython.display import display
from Arm_Lib import Arm_Device

arm = Arm_Device()
time.sleep(0.1)
POSES_FILE = 'poses.json'
MOVE_TIME_MS = 1500
CAMERA_INDEX = 0   # try 1 if the image is black

## Camera (live feed in a background thread)

In [ ]:
cap = cv2.VideoCapture(CAMERA_INDEX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

image_widget = widgets.Image(format='jpeg', width=640, height=480)
latest = {'frame': None}
cam_running = True

def camera_loop():
    while cam_running:
        ok, frame = cap.read()
        if not ok:
            time.sleep(0.05)
            continue
        latest['frame'] = frame.copy()
        h, w = frame.shape[:2]
        cv2.line(frame, (w//2-15, h//2), (w//2+15, h//2), (0,255,0), 1)
        cv2.line(frame, (w//2, h//2-15), (w//2, h//2+15), (0,255,0), 1)
        image_widget.value = cv2.imencode('.jpg', frame)[1].tobytes()
        time.sleep(0.03)

cam_thread = threading.Thread(target=camera_loop, daemon=True)
cam_thread.start()

## Controls

In [ ]:
def load_poses():
    if os.path.exists(POSES_FILE):
        with open(POSES_FILE) as f:
            return json.load(f)
    return {}

poses = load_poses()
names = ['J1 base', 'J2 shoulder', 'J3 elbow', 'J4 wrist', 'J5 wrist rot', 'J6 gripper']
sliders = [widgets.IntSlider(value=90, min=0, max=180, description=n, continuous_update=False) for n in names]

move_btn = widgets.Button(description='Move', button_style='primary')
name_box = widgets.Text(placeholder='pose name (dealer, mycards, ...)')
save_btn = widgets.Button(description='Save pose', button_style='success')
pose_dd = widgets.Dropdown(options=sorted(poses), description='Saved:')
go_btn = widgets.Button(description='Go to saved')
snap_btn = widgets.Button(description='Snap photo')
out = widgets.Output()

def current():
    return [s.value for s in sliders]

def move(q):
    arm.Arm_serial_servo_write6(*q, MOVE_TIME_MS)

def on_move(_):
    move(current())
    with out: print('moved to', current())

def on_save(_):
    n = name_box.value.strip()
    if not n:
        with out: print('enter a name first')
        return
    poses[n] = current()
    with open(POSES_FILE, 'w') as f:
        json.dump(poses, f, indent=2)
    pose_dd.options = sorted(poses)
    with out: print('saved', n, poses[n])

def on_go(_):
    if pose_dd.value:
        for s, a in zip(sliders, poses[pose_dd.value]):
            s.value = a
        move(current())
        with out: print('went to', pose_dd.value, current())

def on_snap(_):
    if latest['frame'] is not None:
        os.makedirs('snaps', exist_ok=True)
        fn = 'snaps/%s_%d.jpg' % (name_box.value.strip() or 'snap', int(time.time()))
        cv2.imwrite(fn, latest['frame'])
        with out: print('wrote', fn)

move_btn.on_click(on_move); save_btn.on_click(on_save)
go_btn.on_click(on_go); snap_btn.on_click(on_snap)

display(widgets.HBox([image_widget, widgets.VBox(sliders + [move_btn])]),
        widgets.HBox([name_box, save_btn, snap_btn]),
        widgets.HBox([pose_dd, go_btn]), out)

## Saved poses

In [ ]:
for n, q in poses.items():
    print('%-10s %s' % (n, q))

## Cleanup (run when done, releases the camera)

In [ ]:
cam_running = False
time.sleep(0.2)
cap.release()